In [1]:
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

run_dir = r"C:\Users\X\Desktop\X\ALTA\runs\20260922-221848_roberta-base"
model_out_dir = os.path.join(run_dir, "final_models")
test_csv = r"C:\Users\X\Desktop\X\ALTA\sample_test\answer.csv"
out_csv = os.path.join(run_dir, "answer.csv")
max_length = 256
batch_size = 32

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}

def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

def predict_task(task, texts, device):
    model_path = os.path.join(model_out_dir, task)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()
    enc = tokenizer(list(texts), truncation=True, max_length=max_length)
    ds = TextDataset(enc)
    collator = DataCollatorWithPadding(tokenizer)
    args = TrainingArguments(output_dir=os.path.join(run_dir, "inference_scratch", task), per_device_eval_batch_size=batch_size, report_to=[])
    trainer = Trainer(model=model, args=args, data_collator=collator)
    preds = trainer.predict(ds)
    return np.argmax(preds.predictions, axis=-1)

device = get_device()
test_df = pd.read_csv(test_csv)
texts = test_df["text"].astype(str).tolist()
answer_df = test_df[["source", "variety", "text"]].copy()
for task in ["sentiment", "sarcasm"]:
    answer_df[task] = predict_task(task, texts, device)
answer_df.to_csv(out_csv, index=False)
answer_df.head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,source,variety,text,sentiment,sarcasm
0,google,en-AU,"Lovely flavours, but a lot of food. Come hungr...",1,0
1,google,en-AU,"Such fresh and flavoursome meals, very quick s...",1,0
2,google,en-AU,"Dissapointed coz I PAY for 8 chicken tender, W...",0,1
3,google,en-AU,Came here with a large party on an already bus...,1,0
4,google,en-AU,On Saturday the at the Ballarat store dinner t...,0,0
